# 12 · Ficha de probabilidad

**Qué calcula.** La probabilidad de que un día de temporada de lluvias sea un **día con huaico catalogado** en Chosica, Chaclacayo, Santa Eulalia o Ricardo Palma, dadas las condiciones del día (lluvia, humedad del suelo, caudal, temperatura del mar, calor en altura, humedad y régimen El Niño), con su intervalo, su fiabilidad histórica y las condiciones que más empujan hacia arriba. Y, por quebrada, la probabilidad de que aparezca en el catálogo ese día.

> **No es una alerta ni una predicción para decidir evacuaciones.** Es la frecuencia histórica de días con huaico *catalogado* bajo condiciones parecidas. El catálogo está incompleto y registra sobre todo donde hay gente; no es un peligro absoluto, ni una probabilidad anual, ni la de una quebrada en particular. Ante una emergencia llama al 115 (INDECI).

## Pre-registro (escrito antes de mirar ningún resultado)

**Datos.** ERA5 en 6 puntos (3 por la API de Open-Meteo y 3 de sus archivos en AWS), temperatura del mar NOAA OISST, caudal GloFAS, ICEN oficial y el catálogo de 148 fechas (días con huaico: 119 en 6 473 días de temporada, 1984-2026).

**Modelos.**
- *Completo*: regresión logística con penalización L2 sobre las 118 columnas del capítulo 05 («+Niño»).
- *Reducido*: igual, pero dentro de cada conjunto de entrenamiento se conservan las 2 columnas de cada familia con mayor |AUC − 0,5| (la familia ICEN conserva sus 2). Es el modelo principal si su log-loss fuera de muestra no supera en más de 0,001 al del completo.
- La penalización C se elige dentro de cada entrenamiento entre (0,003; 0,01; 0,03; 0,1; 0,3) con 4 pliegues internos agrupados por temporada.

**Validación.** Siempre fuera de muestra: se deja fuera una temporada entera (43 pliegues). Sin 70/30 aleatorio.

**Recalibración.** Platt (2 parámetros) ajustada solo con los logits fuera de muestra de las *otras* temporadas. La versión con ajuste de régimen (agrega «ICEN medio causal de los 3 meses previos ≥ 0,4») solo se adopta si baja el log-loss fuera de muestra al menos 1 % y acerca a 1 la razón observado/predicho de temporadas Niño y no Niño.

**Intervalos.** 500 remuestreos de temporadas completas (la unidad que conserva los racimos), cada uno reajustado con su selección de columnas; el intervalo del 80 % de un día pasado usa solo los remuestreos que no vieron su temporada.

**Bandas de fiabilidad fijas:** 0,5 %, 1 %, 2 %, 5 %, 10 % y 25 %.

**Hipótesis.**
- **P1** el modelo reducido no es peor que el completo;
- **P2** la recalibración mejora la fiabilidad (Brier, log-loss);
- **P3** el ajuste de régimen corrige el desbalance Niño / no Niño del capítulo 05 (1,29 y 0,74);
- **P4** la probabilidad sigue calibrada por era y sin los tres grandes racimos (1983, 1998, 2017);
- **P5** las familias que más empujan en los días con huaico son estables entre remuestreos;
- **P6** el producto `P(día) × π_quebrada` ordena los pares (quebrada, día) mejor que solo el día.

No se ajusta ningún umbral hasta acertar el pasado.

In [ ]:
import os
from pathlib import Path

import numpy as np
import pandas as pd

from illapa_research import gpu, probabilidad as pr

RAIZ = Path("..") if Path("../datos").exists() else Path(".")
N_BOOT = int(os.environ.get("ILLAPA_N_BOOT", "500"))
print("dispositivo:", gpu.dispositivo(), "| remuestreos:", N_BOOT)
R = pr.correr(RAIZ, n_boot=N_BOOT)

## 1. Completo contra reducido (P1)

Probabilidades recalibradas fuera de muestra, 43 temporadas dejadas fuera.

In [ ]:
R['tabla_modelos'].round(4)

In [ ]:
print('modelo principal:', R['primario']); R['c_elegidas'].value_counts().sort_index()

## 2. Calibración y fiabilidad (P2, P3)

In [ ]:
glob = pd.Series(R["global"]).round(4)
glob

In [ ]:
pd.DataFrame({"sin ajuste de régimen": R["razones_plano"], "con ajuste de régimen": R["razones_regimen"]}).round(2)

In [ ]:
print(f"mejora del log-loss con el ajuste de régimen: {R['mejora_regimen']:.2%}; adoptado: {R['usa_regimen']}")
R["fiabilidad"].round(4)

Poder de resolución por banda fija: cuántos días caen en cada banda y qué fracción tuvo huaico, con el intervalo del 80 % que remuestrea temporadas enteras.

In [ ]:
R["bandas"][["banda", "dias", "con_huaico", "fraccion", "ic80_temporadas_bajo", "ic80_temporadas_alto", "prob_media"]].round(3)

## 3. Era, régimen y sin los grandes racimos (P4)

In [ ]:
R['por_grupo'].round(3)

In [ ]:
pd.Series(R['sin_racimos_reentrenado']).round(4)

**Exploratorio, no pre-registrado.** La razón observado/predicho cambia con la era (por debajo de 1 antes de 2009 y por encima después): el catálogo está mucho más completo desde 2009. Calibrar solo con las temporadas recientes (dejando una temporada fuera dentro de ellas) es lo que mejor representa la cobertura actual.

In [ ]:
ex = pr.calibracion_desde(R, 2009)
print(ex["dias"], "días,", ex["con_huaico"], "con huaico")
display(pd.DataFrame({"calibrada con todas las eras": ex["plano"], "calibrada solo con 2009+": ex["calibrado_reciente"]}).round(4))
display(ex["bandas"][["banda", "dias", "con_huaico", "fraccion", "ic80_bajo", "ic80_alto", "prob_media"]].round(3))

## 4. Qué empuja hacia arriba (P5)

Aporte medio por familia, en escala logit respecto a un día promedio de temporada, en días con huaico y en los demás, y en qué fracción de los 500 modelos remuestreados la familia empuja más en los días con huaico.

In [ ]:
R['aportes'].round(3)

## 5. Por quebrada (P6)

`P(quebrada i el día t) = P(día t) × π_i`. Validación dejando una temporada fuera para `π_i`: AP, AUC y log-loss sobre todos los pares (quebrada, día).

In [ ]:
R['pares'].round(4)

In [ ]:
pi = R["pi"]
tabla = pi.pivot(index="slug", columns="temporadas", values="pi")
tabla["dias_con_historial"] = pi[pi.temporadas == "todas"].set_index("slug")["dias_activa"]
tabla.round(3).sort_values("todas", ascending=False)

## 6. Retrospectiva

Los días con huaico más probables, los menos probables, y los días sin huaico que el modelo consideró más probables (falsas alarmas del capítulo 07).

In [ ]:
y, p = R["y"], R["p_oof"]
con = np.flatnonzero(y == 1)
sin = np.flatnonzero(y == 0)
print("Días con huaico con mayor probabilidad")
display(pr.retrospectiva(R, con[np.argsort(-p[con])[:8]]).round(3))
print("Días con huaico con menor probabilidad")
display(pr.retrospectiva(R, con[np.argsort(p[con])[:6]]).round(4))
print("Días sin huaico catalogado con mayor probabilidad")
display(pr.retrospectiva(R, sin[np.argsort(-p[sin])[:8]]).round(3))

## 7. Temporadas tipo

Una temporada Niño con muchos huaicos (2017), una Niño casi sin huaicos (2016) y una sin Niño (2018): probabilidad diaria esperada y cuántos días caen en cada banda.

In [ ]:
for t in (2017, 2016, 2018):
    r_ = pr.temporada_tipo(R, t)
    print(f"{t}: {r_['dias']} días, {r_['dias_con_huaico']} con huaico, {r_['dias_esperados']:.1f} esperados, p máxima {r_['p_maxima']:.2f}, mediana {r_['p_mediana']:.4f}")
    display(r_["dias_por_banda"])

## 8. La ficha de un día

Ejemplo con un día de huaico (2017-03-16) y uno sin huaico (2017-04-02). **Ojo:** la ficha usa el modelo final, entrenado con todos los días; para una fecha pasada es «dentro de muestra» y sale más alta de lo que fue en realidad fuera de muestra (compara con la retrospectiva de arriba). Para una fecha nueva es la ficha que corresponde.

In [ ]:
import json
mod = R["modelo"]
X = R["X"]
for f in ("2017-03-16", "2017-04-02"):
    out = pr.ficha(mod, X.loc[f])
    print(f, "->", round(out["probabilidad"], 4), "IC80", tuple(round(v, 4) for v in out["intervalo_80"]), out["banda_de_fiabilidad"]["banda"],
          "| días históricos en la banda:", out["banda_de_fiabilidad"]["dias_historicos"], "con huaico:", out["banda_de_fiabilidad"]["con_huaico"])
    for fa in out["factores_favorables"]:
        print("   -", fa["texto"], f"(aporte {fa['aporte_logit']:+.2f}, estabilidad {fa['estabilidad']:.0%})")
    print("   quebradas más probables:", [(q["nombre"], round(q["probabilidad"], 3), q["evidencia"]) for q in out["quebradas"][:4]])
print(out["limites"])

## Qué NO garantiza esta probabilidad

- No es la probabilidad de que ocurra un huaico: es la de que un día quede **en el catálogo**. Lo no registrado (1999-2008, quebradas sin gente, eventos sin daños) cuenta como «no huaico».
- Con 119 días positivos y 43 temporadas, los intervalos son anchos y las bandas altas tienen pocos días; sin 1983, 1998 y 2017 la habilidad baja.
- Las condiciones favorables son asociaciones con el catálogo, no causas.
- Las probabilidades por quebrada heredan el sesgo de reporte (más edificios, más reportes) y solo existen para las 20 con historial ubicado.
- Depende de ERA5 en 6 puntos y del ICEN: un cambio de fuente cambia los números.